<a href="https://colab.research.google.com/github/BeatrizGiacon/Projeto-Pratico-PLN-/blob/fix-rag-tool/projeto_pln.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto de assistente de pedido do Mc Donalds
Um assistente conversacional em português que ajuda a montar um pedido do McDonald's. Ele entende restrições escritas livremente ("frango com baixo sódio", "eu não como carne de vaca"). Se o usuário não disser nenhuma, o assistente pergunta. Os produtos e valores nutricionais vêm do `menu.csv`, não da memória do modelo.

**Limites dos dados.** O `menu.csv` (carregado da branch `main`) é um conjunto histórico do cardápio dos **EUA**. Ele traz nome, categoria, porção e valores nutricionais, mas **não** traz ingredientes, alergênicos nem informação sobre contaminação cruzada. Por isso:

- restrições como "não como carne bovina" são aplicadas de forma **conservadora**, usando apenas o que o nome e a categoria do produto indicam. Um produto sem indicação no nome **não** é declarado livre do ingrediente; ele aparece como "composição não verificada";
- o assistente **nunca** afirma que um item é seguro para alergias ou intolerâncias.

**Como executar.** Rode as células em ordem (Ambiente de execução → Executar tudo). A conversa **não** começa sozinha: ao final, execute `conversar()` na última célula.

### Instalações

In [24]:
!pip install langchain langchain_openai langchain-chroma langgraph nltk

### Chave e modelo
A chave é lida, nesta ordem, da variável de ambiente `OPENAI_API_KEY`, dos *Secrets* do Colab (ícone de chave na barra lateral) ou digitada com `getpass`. A chave nunca é impressa.

O modelo é usado **apenas** para interpretar a mensagem. Comandos inequívocos (ver pedido, adicionar/remover por ID, perguntas de nutriente sobre um item) são resolvidos localmente e continuam funcionando se o modelo falhar.

In [16]:
# @colab-only: pede credencial interativamente; os testes automatizados não executam esta célula.
import os
from getpass import getpass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except Exception:
        OPENAI_API_KEY = None
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("OPENAI_API_KEY: ")

OPENAI_API_KEY: ··········


In [17]:
# @colab-only: cria o cliente do modelo e faz UMA chamada de teste (consome créditos).
from langchain_openai import ChatOpenAI

modelo = ChatOpenAI(model="gpt-6-luna", api_key=OPENAI_API_KEY, timeout=60, max_retries=1)
try:
    print(modelo.invoke("Responda apenas 'ok' se estiver funcionando.").content)
except Exception as erro:
    print("Aviso: o modelo não respondeu:", type(erro).__name__, "- confira a chave e o saldo da conta antes de continuar.")

ok


### Carregar o corpus
Baixa o `menu.csv` da branch `main` (fonte compartilhada do grupo) e valida se o arquivo não está vazio.

In [18]:
import requests
import pandas as pd
import numpy as np
from io import StringIO

URL_MENU = "https://raw.githubusercontent.com/BeatrizGiacon/Projeto-Pratico-PLN-/main/menu.csv"

def carregar_menu_bruto(url=URL_MENU, timeout=30):
    """Baixa o CSV e devolve o DataFrame bruto. Falha com mensagem clara se o arquivo estiver vazio."""
    resposta = requests.get(url, timeout=timeout)
    resposta.raise_for_status()
    texto = resposta.content.decode("utf-8-sig")
    if not texto.strip():
        raise ValueError("O menu.csv baixado está vazio.")
    bruto = pd.read_csv(StringIO(texto))
    if bruto.empty:
        raise ValueError("O menu.csv não tem linhas de dados.")
    return bruto

df_bruto = carregar_menu_bruto()
print(df_bruto.shape)
df_bruto.head()

(260, 24)


,Category,Item,Serving Size,Calories,Calories from Fat,Total Fat,Total Fat (% Daily Value),Saturated Fat,Saturated Fat (% Daily Value),Trans Fat,...,Carbohydrates,Carbohydrates (% Daily Value),Dietary Fiber,Dietary Fiber (% Daily Value),Sugars,Protein,Vitamin A (% Daily Value),Vitamin C (% Daily Value),Calcium (% Daily Value),Iron (% Daily Value)
0,Breakfast,Egg McMuffin,4.8 oz (136 g),300,120,13.0,20,5.0,25,0.0,...,31,10,4,17,3,17,10,0,25,15
1,Breakfast,Egg White Delight,4.8 oz (135 g),250,70,8.0,12,3.0,15,0.0,...,30,10,4,17,3,18,6,0,25,8
2,Breakfast,Sausage McMuffin,3.9 oz (111 g),370,200,23.0,35,8.0,42,0.0,...,29,10,4,17,2,14,8,0,25,10
3,Breakfast,Sausage McMuffin with Egg,5.7 oz (161 g),450,250,28.0,43,10.0,52,0.0,...,30,10,4,17,2,21,15,0,30,15
4,Breakfast,Sausage McMuffin with Egg Whites,5.7 oz (161 g),400,210,23.0,35,8.0,42,0.0,...,30,10,4,17,2,21,6,0,25,10


In [19]:
RENOMEAR = {
    "Category": "categoria", "Item": "item", "Serving Size": "porcao_texto",
    "Calories": "calorias", "Total Fat": "gordura_g", "Saturated Fat": "gordura_sat_g",
    "Cholesterol": "colesterol_mg", "Sodium": "sodio_mg", "Carbohydrates": "carbo_g",
    "Sugars": "acucar_g", "Protein": "proteina_g", "Calcium (% Daily Value)": "calcio_vd_pct",
}
COLUNAS_NUMERICAS = ["calorias", "gordura_g", "gordura_sat_g", "colesterol_mg", "sodio_mg",
                     "carbo_g", "acucar_g", "proteina_g", "calcio_vd_pct"]

def preparar_menu(bruto):
    """Seleciona e renomeia colunas, valida tipos e cria id_menu = posição da linha (0..n-1).

    Valores numéricos ausentes, não numéricos, negativos ou infinitos viram NaN
    ("não informado"); eles nunca são tratados como zero.
    """
    if bruto is None or len(bruto) == 0:
        raise ValueError("menu.csv sem linhas.")
    faltantes = [c for c in RENOMEAR if c not in bruto.columns]
    if faltantes:
        raise ValueError(f"Colunas ausentes no menu.csv: {faltantes}")
    tabela = bruto[list(RENOMEAR)].rename(columns=RENOMEAR).copy()
    tabela = tabela[tabela["item"].notna()].copy()
    tabela["item"] = tabela["item"].astype(str).str.replace(r"[®™]", "", regex=True).str.strip()
    tabela["categoria"] = tabela["categoria"].fillna("Sem categoria").astype(str).str.strip()
    tabela["porcao_texto"] = tabela["porcao_texto"].fillna("").astype(str).str.strip()
    tabela = tabela[tabela["item"] != ""].copy()
    for coluna in COLUNAS_NUMERICAS:
        valores = pd.to_numeric(tabela[coluna], errors="coerce").astype(float)
        tabela[coluna] = valores.where(np.isfinite(valores) & (valores >= 0))
    if tabela.empty:
        raise ValueError("menu.csv não contém itens válidos.")
    tabela = tabela.reset_index(drop=True)
    tabela.insert(0, "id_menu", range(len(tabela)))
    # Peso/volume da porção quando o texto informa; bebidas em "fl oz" ficam sem valor.
    tabela["porcao_g"] = tabela["porcao_texto"].str.extract(r"\((\d+(?:\.\d+)?) g\)", expand=False).astype(float)
    tabela["porcao_ml"] = tabela["porcao_texto"].str.extract(r"\((\d+(?:\.\d+)?) ml\)", expand=False).astype(float)
    return tabela

df = preparar_menu(df_bruto)
print("Itens:", len(df), "| nomes repetidos:", int(df["item"].duplicated().sum()),
      "| valores numéricos não informados:", int(df[COLUNAS_NUMERICAS].isna().sum().sum()))
df.head()

Itens: 260 | nomes repetidos: 0 | valores numéricos não informados: 0


,id_menu,categoria,item,porcao_texto,calorias,gordura_g,gordura_sat_g,colesterol_mg,sodio_mg,carbo_g,acucar_g,proteina_g,calcio_vd_pct,porcao_g,porcao_ml
0,0,Breakfast,Egg McMuffin,4.8 oz (136 g),300.0,13.0,5.0,260.0,750.0,31.0,3.0,17.0,25.0,136.0,NaN
1,1,Breakfast,Egg White Delight,4.8 oz (135 g),250.0,8.0,3.0,25.0,770.0,30.0,3.0,18.0,25.0,135.0,NaN
2,2,Breakfast,Sausage McMuffin,3.9 oz (111 g),370.0,23.0,8.0,45.0,780.0,29.0,2.0,14.0,25.0,111.0,NaN
3,3,Breakfast,Sausage McMuffin with Egg,5.7 oz (161 g),450.0,28.0,10.0,285.0,860.0,30.0,2.0,21.0,30.0,161.0,NaN
4,4,Breakfast,Sausage McMuffin with Egg Whites,5.7 oz (161 g),400.0,23.0,8.0,50.0,880.0,30.0,2.0,21.0,25.0,161.0,NaN


### Preparar o texto do menu

Os nomes dos itens ("Big Mac", "Chicken McNuggets") são o único texto do corpus. Este passo segue o tratamento de:

- Normalização (minúsculas, sem acentos) e tokenização.
- Remover palavras vazias como "the" e "of" (stopwords) e faz lematização.
- Criar etiquetas úteis.

Se os recursos do NLTK não puderem ser baixados, a célula usa uma lista mínima de stopwords e segue sem lematização (os tokens são apenas ilustrativos; a busca usa TF-IDF).

In [20]:
import re, unicodedata

def normalizar(texto):
    """Minúsculas, sem acentos e com espaços simples. None/NaN viram texto vazio."""
    if texto is None or (isinstance(texto, float) and np.isnan(texto)):
        return ""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", t).strip()

try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    for recurso in ("stopwords", "wordnet", "omw-1.4"):
        nltk.download(recurso, quiet=True)
    stop_en = set(stopwords.words("english"))
    lematizador = WordNetLemmatizer()
    lematizador.lemmatize("tests")
except Exception as erro:
    print("NLTK indisponível (", type(erro).__name__, "): usando stopwords mínimas e sem lematização.")
    stop_en = {"the", "of", "and", "with", "a", "an", "in", "on", "for", "to"}
    lematizador = None

def preprocessar_nome(texto):
    """Normaliza, tokeniza, remove stopwords e lematiza um nome de item."""
    tokens = [t for t in re.findall(r"[a-z]+", normalizar(texto)) if t not in stop_en and len(t) > 2]
    return [lematizador.lemmatize(t) for t in tokens] if lematizador else tokens

df["tokens"] = df["item"].apply(preprocessar_nome)
df[["item", "tokens"]].head(10)

,item,tokens
0,Egg McMuffin,"[egg, mcmuffin]"
1,Egg White Delight,"[egg, white, delight]"
2,Sausage McMuffin,"[sausage, mcmuffin]"
3,Sausage McMuffin with Egg,"[sausage, mcmuffin, egg]"
4,Sausage McMuffin with Egg Whites,"[sausage, mcmuffin, egg, white]"
5,Steak & Egg McMuffin,"[steak, egg, mcmuffin]"
6,"Bacon, Egg & Cheese Biscuit (Regular Biscuit)","[bacon, egg, cheese, biscuit, regular, biscuit]"
7,"Bacon, Egg & Cheese Biscuit (Large Biscuit)","[bacon, egg, cheese, biscuit, large, biscuit]"
8,"Bacon, Egg & Cheese Biscuit with Egg Whites (R...","[bacon, egg, cheese, biscuit, egg, white, regu..."
9,"Bacon, Egg & Cheese Biscuit with Egg Whites (L...","[bacon, egg, cheese, biscuit, egg, white, larg..."


### Documentação para busca semântica
Transforma cada linha em um Document do LangChain, com um texto descritivo e metadados (incluindo `id_menu`). É a base do RAG.

In [21]:
from langchain_core.documents import Document

def _valor_txt(x, unidade=""):
    return "não informado" if pd.isna(x) else f"{x:.0f}{(' ' + unidade) if unidade else ''}"

def descrever(linha):
    return (f"{linha.item} (categoria: {linha.categoria}). Calorias: {_valor_txt(linha.calorias)}; "
            f"sódio: {_valor_txt(linha.sodio_mg, 'mg')}; gordura: {_valor_txt(linha.gordura_g, 'g')}; "
            f"açúcar: {_valor_txt(linha.acucar_g, 'g')}; proteína: {_valor_txt(linha.proteina_g, 'g')}.")

def criar_documentos(tabela):
    return [Document(page_content=descrever(l),
                     metadata={"id_menu": int(l.id_menu), "item": l.item, "categoria": l.categoria})
            for l in tabela.itertuples()]

documentos = criar_documentos(df)
print(len(documentos), "documentos. Exemplo:", documentos[0].page_content)

260 documentos. Exemplo: Egg McMuffin (categoria: Breakfast). Calorias: 300; sódio: 750 mg; gordura: 13 g; açúcar: 3 g; proteína: 17 g.


### Índice vetorial (RAG)
Cada item do menu vira um vetor (*embedding*). O Chroma permite buscar por significado, como "algo leve de frango", e limitar a busca aos IDs permitidos pelas restrições do cliente.

In [26]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)
vectorstore = InMemoryVectorStore.from_documents(
    documentos, embeddings,
    ids=[str(d.metadata["id_menu"]) for d in documentos],  # ids fixos: reexecutar a célula não duplica
)

for d in vectorstore.similarity_search("light chicken sandwich", k=3):
    print(d.page_content)

Premium Crispy Chicken Club Sandwich (categoria: Chicken & Fish). Calorias: 670; sódio: 1410 mg; gordura: 33 g; açúcar: 11 g; proteína: 36 g.
Premium Grilled Chicken Club Sandwich (categoria: Chicken & Fish). Calorias: 510; sódio: 1250 mg; gordura: 20 g; açúcar: 9 g; proteína: 40 g.
Premium Grilled Chicken Classic Sandwich (categoria: Chicken & Fish). Calorias: 350; sódio: 820 mg; gordura: 9 g; açúcar: 8 g; proteína: 28 g.


### Estado do pedido e regra única de bloqueio
O `menu.csv` não traz ingredientes nem alérgenos. A única regra feita em código é conservadora: uma proteína excluída bloqueia os itens cujo nome a indica e as categorias compartilhadas ("Beef & Pork", "Chicken & Fish"). A função `motivo_bloqueio` é usada por todas as ferramentas, então nenhuma recomendação ou adição ignora uma restrição.

In [27]:
import re
from typing import Literal, Optional
from uuid import uuid4

Proteina = Literal["bovino", "frango", "porco", "peixe"]

# Termos (em inglês) que, no NOME do item, indicam a proteína.
PADROES_PROTEINA = {
    "bovino": re.compile(r"\b(beef|hamburger|cheeseburger|burger|big mac|angus|quarter pounder|mcdouble|steak)\b", re.I),
    "frango": re.compile(r"\b(chicken|mcnuggets?|mcchicken)\b", re.I),
    "porco": re.compile(r"\b(bacon|sausage|ham|pork|mcrib)\b", re.I),
    "peixe": re.compile(r"\b(fish|filet-o-fish)\b", re.I),
}
# A categoria não permite separar as duas proteínas, então bloqueamos as duas.
CATEGORIAS_COMPARTILHADAS = {
    "bovino": {"Beef & Pork"}, "porco": {"Beef & Pork"},
    "frango": {"Chicken & Fish"}, "peixe": {"Chicken & Fish"},
}
LIMITES = {"sodio_max_mg": "sodio_mg", "calorias_max": "calorias", "acucar_max_g": "acucar_g"}
AVISO_COMPOSICAO = ("O menu.csv não traz ingredientes nem alérgenos: a análise usa apenas nome e categoria. "
                    "Um item não bloqueado NÃO é garantidamente seguro; confirme a composição oficial.")

def novo_estado():
    return {"excluir_proteinas": set(), "evitar_termos": set(), "sodio_max_mg": None,
            "calorias_max": None, "acucar_max_g": None, "proteina_min_g": None,
            "restricoes_respondidas": False}

estado = novo_estado()   # restrições ativas do cliente
pedido = []              # lista de id_menu dos itens pedidos

def motivo_bloqueio(linha):
    """Devolve o motivo pelo qual o item viola as restrições ativas, ou None se ele é permitido."""
    nome = linha["item"]
    for p in estado["excluir_proteinas"]:
        if linha["categoria"] in CATEGORIAS_COMPARTILHADAS[p]:
            return f"a categoria '{linha['categoria']}' pode conter {p}"
        if PADROES_PROTEINA[p].search(nome):
            return f"o nome indica {p}"
    for termo in estado["evitar_termos"]:
        if termo.lower() in nome.lower():
            return f"o nome contém '{termo}'"
    for chave, coluna in LIMITES.items():
        # a comparação falha para NaN, então um valor não informado também bloqueia
        if estado[chave] is not None and not linha[coluna] <= estado[chave]:
            return f"{coluna} acima de {estado[chave]:g} (ou não informado)"
    minimo = estado["proteina_min_g"]
    if minimo is not None and not linha["proteina_g"] >= minimo:
        return f"proteína abaixo de {minimo:g} g (ou não informada)"
    return None

def permitidos():
    """Itens do menu que respeitam todas as restrições ativas."""
    return df[[motivo_bloqueio(l) is None for _, l in df.iterrows()]]

COLS = ["id_menu", "item", "categoria", "calorias", "sodio_mg", "acucar_g", "proteina_g"]

def registros(tabela):
    """Converte linhas em dicionários, com None no lugar de NaN (JSON válido para o modelo)."""
    sub = tabela[COLS]
    return sub.astype(object).where(sub.notna(), None).to_dict("records")

In [ ]:
# @colab-only: exemplo com chamada real ao modelo (consome créditos).
try:
    print(interpretar_llm("Quero frango com pouco sódio, até 500 mg").model_dump())
except Exception as erro:
    print("Exemplo não executado:", type(erro).__name__)

### Ferramentas (extração de informação via argumentos)
Cada `@tool` é uma função que o modelo pode chamar. Os argumentos de `registrar_restricoes` seguem um esquema Pydantic validado (limites não negativos, sem NaN), que é a extração de informação: o modelo converte "não como carne de vaca" em `excluir_proteinas=["bovino"]` e traduz termos como "leite" para `milk`. A docstring e as descrições dos campos são o que o modelo lê para decidir como usar cada ferramenta.

In [28]:
from pydantic import BaseModel, ConfigDict, Field
from langchain_core.tools import tool

class Restricoes(BaseModel):
    model_config = ConfigDict(allow_inf_nan=False)
    excluir_proteinas: list[Proteina] = Field(
        default_factory=list, description="Carnes que o cliente não come. 'Não como carne de vaca' = ['bovino'].")
    liberar_proteinas: list[Proteina] = Field(
        default_factory=list, description="Carnes que o cliente voltou a aceitar.")
    evitar_termos: list[str] = Field(
        default_factory=list,
        description="Palavras EM INGLÊS dos nomes dos itens a evitar, para alergias e intolerâncias. Ex.: milk, cheese, egg, ice cream.")
    sodio_max_mg: Optional[float] = Field(
        default=None, ge=0, description="Máximo de sódio por item, em mg, INCLUSIVO. 'Menos de X' = X-1.")
    calorias_max: Optional[float] = Field(
        default=None, ge=0, description="Máximo de calorias por item, INCLUSIVO. 'Menos de X' = X-1.")
    acucar_max_g: Optional[float] = Field(
        default=None, ge=0, description="Máximo de açúcar por item, em g, INCLUSIVO. 'Menos de X' = X-1.")
    proteina_min_g: Optional[float] = Field(
        default=None, ge=0, description="Mínimo de proteína por item, em g.")
    sem_restricoes: bool = Field(default=False, description="True se o cliente disse que não tem restrições.")
    reiniciar: bool = Field(default=False, description="True para apagar todas as restrições antes de aplicar as novas.")

@tool(args_schema=Restricoes)
def registrar_restricoes(excluir_proteinas=(), liberar_proteinas=(), evitar_termos=(),
                         sodio_max_mg=None, calorias_max=None, acucar_max_g=None,
                         proteina_min_g=None, sem_restricoes=False, reiniciar=False) -> dict:
    """Registra ou altera as restrições alimentares e nutricionais do cliente. Chame sempre que o cliente mencionar o que evita ou um limite."""
    if reiniciar:
        estado.clear()
        estado.update(novo_estado())
    estado["excluir_proteinas"] |= set(excluir_proteinas)
    estado["excluir_proteinas"] -= set(liberar_proteinas)
    estado["evitar_termos"] |= {t.strip().lower() for t in evitar_termos if t.strip()}
    numericos = {"sodio_max_mg": sodio_max_mg, "calorias_max": calorias_max,
                 "acucar_max_g": acucar_max_g, "proteina_min_g": proteina_min_g}
    for chave, valor in numericos.items():
        if valor is not None:
            estado[chave] = valor
    if sem_restricoes or excluir_proteinas or evitar_termos or any(v is not None for v in numericos.values()):
        estado["restricoes_respondidas"] = True
    conflitos = [int(i) for i in pedido if motivo_bloqueio(df.iloc[i])]
    resumo = {k: (sorted(v) if isinstance(v, set) else v) for k, v in estado.items()}
    return {"restricoes_ativas": resumo, "itens_do_pedido_em_conflito": conflitos, "aviso": AVISO_COMPOSICAO}

@tool
def filtrar_menu(categoria: str = "", palavra_chave: str = "",
                 ordenar_por: Literal["calorias", "sodio_mg", "acucar_g", "proteina_g"] = "calorias",
                 decrescente: bool = False, limite: int = 6) -> dict:
    """Lista itens do menu que respeitam as restrições ativas. categoria e palavra_chave são EM INGLÊS
    (categorias: Breakfast, Beef & Pork, Chicken & Fish, Salads, Snacks & Sides, Desserts, Beverages,
    Coffee & Tea, Smoothies & Shakes). Deixe vazio para não filtrar. Use ordenar_por='sodio_mg' para baixo sódio."""
    r = permitidos()
    if categoria:
        r = r[r["categoria"].str.contains(categoria, case=False, regex=False)]
    if palavra_chave:
        achou = r["item"].str.contains(palavra_chave, case=False, regex=False)
        sem = r["item"].str.contains(f"without {palavra_chave}", case=False, regex=False)
        r = r[achou & ~sem]  # "(without Chicken)" não é um item com frango
    r = r.sort_values(ordenar_por, ascending=not decrescente, na_position="last").head(max(1, min(limite, 10)))
    return {"itens": registros(r), "restricoes_respondidas": estado["restricoes_respondidas"],
            "aviso": AVISO_COMPOSICAO}

@tool
def buscar_semantica(consulta: str, limite: int = 5) -> dict:
    """Busca itens por descrição livre (em inglês), apenas entre os que respeitam as restrições ativas.
    Use quando o cliente descrever o que quer sem citar um nome ou categoria."""
    ids = permitidos()["id_menu"].tolist()
    if not ids:
        return {"itens": [], "aviso": "Nenhum item atende às restrições atuais."}
    ids_permitidos = set(ids)
    docs = vectorstore.similarity_search(consulta, k=max(1, min(limite, 10)),
                                         filter=lambda d: d.metadata["id_menu"] in ids_permitidos)
    achados = df[df["id_menu"].isin([d.metadata["id_menu"] for d in docs])]
    return {"itens": registros(achados), "aviso": AVISO_COMPOSICAO}

def _linha_valida(id_menu):
    if isinstance(id_menu, bool) or not isinstance(id_menu, int) or not 0 <= id_menu < len(df):
        return None
    return df.iloc[id_menu]

COLS_DETALHE = COLS + ["gordura_g", "gordura_sat_g", "colesterol_mg", "carbo_g", "calcio_vd_pct", "porcao_texto"]

@tool
def consultar_por_nome(nome: str) -> dict:
    """Procura itens pelo nome (em inglês), IGNORANDO as restrições, e devolve o id_menu e todos os valores
    nutricionais. Use para perguntas sobre um item específico e para descobrir o id_menu de um item citado pelo
    cliente. O campo bloqueado_pelas_restricoes informa se o item violaria as restrições atuais."""
    achados = df[df["item"].str.contains(nome, case=False, regex=False)].head(5)
    if achados.empty:
        return {"itens": [], "aviso": f"Nenhum item com '{nome}' no menu."}
    itens = []
    for _, linha in achados.iterrows():
        dados = linha[COLS_DETALHE].astype(object).where(linha[COLS_DETALHE].notna(), None).to_dict()
        dados["bloqueado_pelas_restricoes"] = motivo_bloqueio(linha)
        itens.append(dados)
    return {"itens": itens, "aviso": AVISO_COMPOSICAO}

@tool
def adicionar_ao_pedido(id_menu: int) -> dict:
    """Adiciona um item ao pedido pelo id_menu. Só chame depois que o cliente confirmar o item.
    A ferramenta recusa itens que violam as restrições ativas e informa o motivo."""
    linha = _linha_valida(id_menu)
    if linha is None:
        return {"erro": f"ID inválido: use um inteiro entre 0 e {len(df) - 1}."}
    motivo = motivo_bloqueio(linha)
    if motivo:
        return {"adicionado": False, "motivo": f"Não adicionei {linha['item']}: {motivo}."}
    if id_menu in pedido:
        return {"adicionado": False, "motivo": f"{linha['item']} já está no pedido."}
    pedido.append(id_menu)
    return {"adicionado": True, "item": linha["item"], "aviso": AVISO_COMPOSICAO}

@tool
def remover_do_pedido(id_menu: int) -> dict:
    """Remove um item do pedido pelo id_menu."""
    if id_menu not in pedido:
        return {"removido": False, "motivo": "Esse ID não está no pedido."}
    pedido.remove(id_menu)
    return {"removido": True}

@tool
def ver_pedido() -> dict:
    """Mostra os itens do pedido, os totais (calorias, sódio, açúcar, proteína) e os itens que conflitam com as restrições atuais."""
    itens = df[df["id_menu"].isin(pedido)]
    return {"itens": registros(itens),
            "totais": {"calorias": float(itens["calorias"].sum()), "sodio_mg": float(itens["sodio_mg"].sum()),
                       "acucar_g": float(itens["acucar_g"].sum()), "proteina_g": float(itens["proteina_g"].sum())},
            "itens_em_conflito_com_restricoes": [int(i) for i in pedido if motivo_bloqueio(df.iloc[i])]}

ferramentas = [registrar_restricoes, filtrar_menu, buscar_semantica, consultar_por_nome,
               adicionar_ao_pedido, remover_do_pedido, ver_pedido]
print([t.name for t in ferramentas])

['registrar_restricoes', 'filtrar_menu', 'buscar_semantica', 'consultar_por_nome', 'adicionar_ao_pedido', 'remover_do_pedido', 'ver_pedido']


### O agente
`create_agent` junta o modelo, as ferramentas e as instruções. O `InMemorySaver` guarda a conversa por `thread_id`. O modelo decide quando chamar cada ferramenta (*tool calling*) e escreve a resposta em português a partir dos dados devolvidos.

In [29]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

INSTRUCOES = """Você é um assistente de pedidos do McDonald's. Responda em português, de forma curta e amigável.
O menu está em inglês e só pode ser consultado pelas ferramentas. Nunca invente itens, IDs ou valores nutricionais.

Fluxo:
1. Quando o cliente disser o que evita ou citar limites (carnes, alergias, sódio, calorias, açúcar, proteína), chame
   registrar_restricoes UMA vez com tudo o que ele disse. Traduza os termos de evitar_termos para inglês.
   'Baixo sódio' sem número: use sodio_max_mg=600 e avise o cliente que assumiu esse limite.
2. Se o cliente ainda não falou de restrições, pergunte antes de recomendar: evita alguma carne? tem alergia ou
   intolerância (como lactose)? quer café da manhã ou sobremesa? limite de sódio, calorias ou açúcar?
   Se ele disser que não tem restrições, chame registrar_restricoes(sem_restricoes=True).
3. Para recomendar, use filtrar_menu (categoria, palavra-chave, ordenação) ou buscar_semantica. Mostre no máximo 3
   opções, com id_menu, calorias e sódio. Se vier lista vazia, explique que as restrições zeraram as opções e sugira relaxar uma.
4. Quando o cliente citar um item pelo nome, ou perguntar sobre um item específico (calorias, sódio...), use
   consultar_por_nome para obter o id_menu e os valores, e traduza o nome para inglês.
   Só chame adicionar_ao_pedido depois que o cliente confirmar. Se a ferramenta recusar, explique o motivo e não insista.
5. Para totais, use ver_pedido. Se houver itens em conflito com as restrições, avise o cliente.
6. Alergias e intolerâncias: o menu não traz alérgenos. Registre os termos, mas NUNCA diga que um item é seguro ou
   livre de um ingrediente; lembre o cliente de conferir a composição oficial."""

agente = create_agent(
    model=modelo,
    tools=ferramentas,
    checkpointer=InMemorySaver(),
    system_prompt=INSTRUCOES,
)

def nova_sessao():
    """Zera restrições e pedido e devolve um novo thread_id para a memória da conversa."""
    estado.clear()
    estado.update(novo_estado())
    pedido.clear()
    return str(uuid4())

### Conversa de demonstração com o modelo real
Conversa roteirizada com o `gpt-6-luna`. A saída mostra cada ferramenta que o modelo chamou, o que comprova que as respostas vêm do corpus. Se a conversa for executada de novo, as respostas podem variar. Salve o notebook só depois de executar a versão final.

In [37]:
def texto_da_resposta(mensagem):
    """Extrai só o texto da resposta. Alguns modelos devolvem uma lista de blocos (raciocínio + texto)."""
    conteudo = mensagem.content
    if isinstance(conteudo, str):
        return conteudo
    return "".join(b if isinstance(b, str) else b.get("text", "")
                   for b in conteudo if isinstance(b, str) or b.get("type") == "text")

def conversar(mensagem, thread_id):
    """Envia uma mensagem ao agente, mostra as ferramentas chamadas neste turno e a resposta."""
    config = {"configurable": {"thread_id": thread_id}}
    saida = agente.invoke({"messages": [{"role": "user", "content": mensagem}]}, config=config)
    mensagens = saida["messages"]
    inicio = max(i for i, m in enumerate(mensagens) if m.type == "human")
    for m in mensagens[inicio:]:
        for chamada in getattr(m, "tool_calls", None) or []:
            print(f"   [ferramenta] {chamada['name']}({chamada['args']})")
    print("Assistente:", texto_da_resposta(mensagens[-1]), "\n")

sessao = nova_sessao()
for fala in ["Oi, quero pedir um lanche",
             "Eu não como carne de vaca e quero pouco sódio",
             "Adicione o segundo ao pedido",
             "Tenho intolerância a lactose. Quero uma sobremesa",
             "Adicione o Big Mac",  # deve ser recusado: o nome indica carne bovina
             "Quantas calorias tem o McChicken?",
             "Mostre meu pedido"]:
    print("Usuário:", fala)
    conversar(fala, sessao)

Usuário: Oi, quero pedir um lanche
Assistente: Claro! Antes de sugerir: você evita alguma carne ou tem alergia/intolerância? Prefere café da manhã ou sobremesa? E tem algum limite de sódio, calorias ou açúcar? 

Usuário: Eu não como carne de vaca e quero pouco sódio
   [ferramenta] registrar_restricoes({'excluir_proteinas': ['bovino'], 'liberar_proteinas': [], 'evitar_termos': [], 'sodio_max_mg': 600, 'calorias_max': None, 'acucar_max_g': None, 'proteina_min_g': None, 'sem_restricoes': False, 'reiniciar': False})
   [ferramenta] buscar_semantica({'consulta': 'low sodium snack, no beef', 'limite': 3})
Assistente: Como você pediu pouco sódio, considerei até **600 mg por item**. Estas opções atendem ao limite e não contêm carne bovina pelo nome:

- **Side Salad** — ID 100: 20 calorias, 10 mg de sódio
- **Premium Southwest Salad (without Chicken)** — ID 87: 140 calorias, 150 mg de sódio
- **Premium Bacon Ranch Salad (without Chicken)** — ID 84: 140 calorias, 300 mg de sódio

Qual você gost

### Sumarização do pedido
Técnica de sumarização: o modelo resume o pedido final e os totais nutricionais.

In [38]:
resumo = modelo.invoke(
    "Resuma em até 3 linhas, em português, este pedido do McDonald's e seus totais nutricionais: "
    + str(ver_pedido.invoke({}))
)
print(texto_da_resposta(resumo))

Salada Premium Southwest sem frango.
Total: 140 kcal, 150 mg de sódio, 6 g de açúcar e 6 g de proteína.


### Conversa interativa
Para conversar à vontade, rode `conversar_interativo()` numa célula nova. Digite `sair` para encerrar. A função só é definida aqui, para não travar o "Executar tudo".

In [39]:
def conversar_interativo():
    sessao = nova_sessao()
    while True:
        texto = input("Você (ou 'sair'): ").strip()
        if texto.lower() in {"", "sair"}:
            break
        conversar(texto, sessao)

In [40]:
conversar_interativo()

Você (ou 'sair'): gostaria de um sorvete
Assistente: Claro! Antes de recomendar: você evita alguma carne ou tem alergia/intolerância? Quer limitar sódio, calorias ou açúcar? 

Você (ou 'sair'): tenho alergia a morango
   [ferramenta] registrar_restricoes({'excluir_proteinas': [], 'liberar_proteinas': [], 'evitar_termos': ['strawberry'], 'sodio_max_mg': None, 'calorias_max': None, 'acucar_max_g': None, 'proteina_min_g': None, 'sem_restricoes': False, 'reiniciar': False})
   [ferramenta] filtrar_menu({'categoria': 'Desserts', 'palavra_chave': '', 'ordenar_por': 'calorias', 'decrescente': False, 'limite': 3})
Assistente: Encontrei o **Kids Ice Cream Cone** (ID 106): **45 calorias** e **20 mg de sódio**. Como o menu não informa alérgenos, confirme a composição oficial antes de consumir, por causa da sua alergia a morango. 

Você (ou 'sair'): quero esse então no sabor chocolate
   [ferramenta] consultar_por_nome({'nome': 'Kids Ice Cream Cone'})
Assistente: O menu só lista **Kids Ice Cream C